# RAW to RGB — Part 2: Debayering

The sensor records only one colour per pixel arranged in a **Bayer pattern**:

```
R G R G ...
G B G B ...
R G R G ...
...........
```

To obtain a full-colour image we need to **interpolate** the two missing colour values at each pixel.
This process is called **debayering** or **demosaicing**.

In this notebook we implement **bilinear debayering** step by step.

In [ ]:
import numpy as np              # needed for image arrays
import HdM as HdM               # a useful function I prepared for you to display images pixel per pixel on your monitor
import imageio.v3 as iio        # Reading images with OpenImageIO
from scipy.ndimage import zoom  # Useful function to scale images

linear2sRGB = lambda x: np.where(x > 0.0031308, 1.055 * np.clip(x, 0, None) ** (1/2.4) - 0.055, 12.92 * x)

data = np.load('20_RawBayerImage_BlackSubtracted.npz')
img_black_subtracted = data['img_black_subtracted']
black_noise_std      = data['black_noise_std']

print(f'Loaded image shape: {img_black_subtracted.shape}')

## Look closely at the Bayer pattern

Zoom into a small crop — you should clearly see the RGGB mosaic.

In [ ]:
stops_above_white = 2
crop = img_black_subtracted[1000:1064, 1000:1064]

HdM.show(linear2sRGB(np.clip(np.repeat(np.repeat(crop, 8, axis=0), 8, axis=1) * 2**stops_above_white, 0, 1)))
print('64×64 crop — individual Bayer pixels visible')


## Split the Bayer plane into 4 sub-images

The RGGB Bayer pattern maps to four sub-sampled images:

| Position | Channel |
|---|---|
| even rows, even cols | c1 |
| odd rows,  even cols | c2 |
| even rows, odd  cols | c3 |
| odd rows,  odd  cols | c4 |

In Python array indexing rows and columns start at 0, so `0::2` selects every second element starting at 0 (even), `1::2` selects odd.

In [ ]:
c1 = img_black_subtracted[0::2, 0::2]   # top-left    of each 2x2 block
c2 = img_black_subtracted[1::2, 0::2]   # bottom-left
c3 = img_black_subtracted[0::2, 1::2]   # top-right
c4 = img_black_subtracted[1::2, 1::2]   # bottom-right

composite_img = np.vstack([np.hstack([c1,c2]),np.hstack([c3,c4])])

HdM.show(linear2sRGB(np.clip(zoom(composite_img, 0.2) * 2**stops_above_white, 0, 1)))
print('Four Bayer sub-images — which two are green? Which one represents red, which one blue?')


## Identify the colour channels

The two green channels should look the most similar to each other.
We compute the **sum of absolute differences** between all channel pairs to find them.

In [ ]:
def sad(a, b):
    """Sum of absolute differences."""
        return  # Hinweis: Calculate the sum of absolute differences

for pair in [('c1','c2'), ('c1','c3'), ('c1','c4'), ('c2','c3'), ('c2','c4'), ('c3','c4')]:
    channels = {'c1': c1, 'c2': c2, 'c3': c3, 'c4': c4}
    print(f'SAD({pair[0]}, {pair[1]}) = {sad(channels[pair[0]], channels[pair[1]]):.0f}')

## Simple 1:4 combine (Canon C300 MkI-style)

The quickest approach: average the two green channels, keep R and B as-is.
This gives a half-resolution colour image.

In [ ]:
# Canon RGGB pattern: c2=R, c1 and c4 = the two greens, c3=B
r =   # Hinweis: Assign the red channel to variable "r"
g =   # Hinweis: Calculate "g" as the mean of both green channels
b =   # Hinweis: Assign the blue channel to variable "b"

rgb_small = np.stack([r, g, b], axis=2)

print('Half-resolution colour image (direct channel combine)')
HdM.show(linear2sRGB(np.clip( zoom( rgb_small, [0.2,0.2,1] ) * 2**stops_above_white, 0, 1)))



## Full-resolution bilinear debayering

To keep full resolution we **interpolate** each missing colour value from its neighbours.
For a RGGB pattern the bilinear interpolation kernels are:

**Red** (known at top-left of each 2×2 block):
```
1/4  1/2  1/4
1/2   1   1/2
1/4  1/2  1/4
```
applied only at non-R positions.

In practice the cleanest implementation uses **convolution with known-pixel masks**.
We follow the standard approach: convolve the full Bayer plane with colour-specific kernels
after masking out the wrong-colour pixels.

In [ ]:
# Crop one row so the image starts with the r g / g b pattern at (0,0)
# (required so our masks align with the correct Bayer colours)
bayer = img_black_subtracted[1:-1, :]   # crop top and bottom row

H, W = bayer.shape

# Build colour masks (True where that colour is actually measured)
# Pattern after crop: row 0 = R G R G …, row 1 = G B G B …
mask_R = np.zeros((H, W), dtype=bool)
mask_G = np.zeros((H, W), dtype=bool)
mask_B = np.zeros((H, W), dtype=bool)

mask_R[0::2, 0::2] = True   # top-left of 2×2
mask_G[0::2, 1::2] = True   # top-right
mask_G[1::2, 0::2] = True   # bottom-left
mask_B[1::2, 1::2] = True   # bottom-right

# Bilinear interpolation kernels
kernel_R = np.array([[1, 2, 1],
                      [2, 4, 2],
                      [1, 2, 1]]) / 4.0

kernel_G = np.array([[0, 1, 0],
                      [1, 4, 1],
                      [0, 1, 0]]) / 4.0

kernel_B = kernel_R.copy()  # same weights, different positions

# Apply: convolve masked values, then normalise by the convolved mask
# (so edges and missing pixels do not shrink the average)

def np_convolve2d(image, kernel, mode='mirror'):
    """Pure NumPy 2D convolution — replaces scipy.ndimage.convolve."""
    kh, kw = kernel.shape
    ph, pw = kh // 2, kw // 2
    pad_mode = 'reflect' if mode == 'mirror' else 'constant'
    img_pad  = np.pad(image, ((ph, ph), (pw, pw)), mode=pad_mode)
    H, W     = image.shape
    out      = np.zeros((H, W))
    for i in range(kh):
        for j in range(kw):
            out += kernel[i, j] * img_pad[i:i+H, j:j+W]
    return out

def bilinear_channel(bayer, mask, kernel):
    """Interpolate one colour channel from the Bayer plane."""
    values = bayer * mask
    weight = np_convolve2d(mask.astype(float), kernel, mode='mirror')
    interp = np_convolve2d(values, kernel, mode='mirror') / np.maximum(weight, 1e-6)
    # Where we have a measured value, use it directly
    return np.where(mask, bayer, interp)

r_full = bilinear_channel(bayer, mask_R, kernel_R)
g_full = bilinear_channel(bayer, mask_G, kernel_G)
b_full = bilinear_channel(bayer, mask_B, kernel_B)

rgb = np.stack([r_full, g_full, b_full], axis=2)


# New code:
import numpy as np
from scipy.ndimage import convolve

# Example: average kernel for interpolating green channel
kernel = np.array([[0, 1, 0],
                   [1, 0, 1],
                   [0, 1, 0]]) / 4.0

filtered = convolve(img.astype(float), kernel)

# Mask: only apply result at every second pixel (e.g. red positions in RGGB)
mask = np.zeros_like(img, dtype=bool)
mask[0::2, 0::2] = True  # every second row and column

result = np.where(mask, filtered, img)
# End new code


stops = 1

HdM.show(linear2sRGB(np.clip(zoom(rgb,[0.2,0.2,1]) * 2**stops, 0, 1)))


## Compare: direct combine vs. bilinear interpolation

We crop a small region to compare resolution and edge sharpness.

In [ ]:
vo = 2260 // 2   # vertical offset (half-res coordinates for rgb_small)
ho = 2780 // 2   # horizontal offset (half-res coordinates for rgb_small)
crop_size = 100
stops = 1

crop_small = rgb_small[vo:vo+crop_size, ho:ho+crop_size, :]
# Upsample for display by nearest-neighbour (zoom=2)
crop_small_up = np.repeat(np.repeat(crop_small, 2, axis=0), 2, axis=1)

crop_full = rgb[vo*2:(vo+crop_size)*2, ho*2:(ho+crop_size)*2, :]

print('4:1 combine (half-res) + upscale on the left compared to bilinear interpolation (full-res) on the right')
HdM.show(np.hstack([crop_small_up,crop_full]))

print('Edges on the right side show zipper artifacts — fixed in next notebook with edge-directed debayering')


## Save result

In [ ]:
np.savez('21_RGB_BilinearDebayer.npz',
         rgb=rgb,
         img_black_subtracted_for_debayering=bayer,
         black_noise_std=black_noise_std)

print('Saved: 21_RGB_BilinearDebayer.npz')